# Engenharia de Dados - Projeto E-Commerce (Camada Gold)
**Objetivo:** Transformar os dados refinados da camada Silver em Data Marts analíticos (Camada Gold) para extração de KPIs de negócio. O foco consiste em consolidar visões de performance comercial e qualidade de atendimento para subsidiar decisões estratégicas da diretoria.

## Configuração do Ambiente
**Objetivo:** Definir o catálogo e o schema gold

In [0]:
# Definindo o catálogo de trabalho
spark.sql("USE CATALOG projeto_medalhao_visagio")

# Garantindo a existência do schema Silver para persistência das tabelas transformadas
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

DataFrame[]

## 1º Projeto: Visão Comercial e Volume de Produtos
**Descrição:** Este projeto foca na consolidação de métricas financeiras e de inventário. 

**Objetivo:** Fornecer à área comercial uma visão clara da evolução das receitas (BRL/USD) e do volume de vendas por categoria.

### Entrega 1 - Tabela Principal (`gold.fat_vendas_comercial`)
**Regras de Negócio Aplicadas:**
* **Granularidade:** Agrupamento por Ano, Mês e Categoria de Produto.
* **Métricas:** Cálculo de pedidos únicos, total de itens, receita convertida e ticket médio.
* **Arredondamento:** Aplicação rigorosa de 2 casas decimais em todos os campos financeiros.

In [0]:
# Carregamento das tabelas necessárias da camada Silver
df_pedidos_total = spark.table("silver.fat_pedido_total")
df_itens_pedidos = spark.table("silver.fat_itens_pedidos")
df_produtos = spark.table("silver.dim_produtos")

In [0]:
from pyspark.sql import functions as F

# Join para consolidar as informações necessárias para os KPIs
df_vendas_completo = df_pedidos_total.join(
    df_itens_pedidos, "id_pedido", "inner"
).join(
    df_produtos, "id_produto", "inner"
)

# Transformações e Agrupamentos conforme requisitos 
# Granularidade: Ano, Mês e Categoria de Produto
df_gold_comercial = df_vendas_completo.groupBy(
    F.year("data_pedido").alias("ano_venda"),
    F.month("data_pedido").alias("mes_venda"),
    F.col("categoria_produto")
).agg(
    F.countDistinct("id_pedido").alias("total_pedidos"),
    F.count("id_item").alias("qtd_itens_vendidos"),
    F.sum("valor_total_pago_brl").alias("receita_total_brl"),
    F.sum("valor_total_pago_usd").alias("receita_total_usd")
)

# Cálculo do Ticket Médio e Arredondamentos
df_gold_comercial = df_gold_comercial.withColumn(
    "ticket_medio_brl", 
    F.col("receita_total_brl") / F.col("total_pedidos")
).select(
    "ano_venda",
    "mes_venda",
    "categoria_produto",
    "total_pedidos",
    "qtd_itens_vendidos",
    F.round("receita_total_brl", 2).alias("receita_total_brl"),
    F.round("receita_total_usd", 2).alias("receita_total_usd"),
    F.round("ticket_medio_brl", 2).alias("ticket_medio_brl")
)

# Escrita da tabela na camada Gold
df_gold_comercial.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fat_vendas_comercial")

display(df_gold_comercial.limit(5))

ano_venda,mes_venda,categoria_produto,total_pedidos,qtd_itens_vendidos,receita_total_brl,receita_total_usd,ticket_medio_brl
2017,6,informatica_acessorios,219,261,58864.94,17898.04,268.79
2018,3,cama_mesa_banho,670,798,118984.21,36250.39,177.59
2017,3,cama_mesa_banho,249,289,41668.27,13316.55,167.34
2017,8,informatica_acessorios,297,350,62791.82,19914.57,211.42
2017,7,eletronicos,87,97,12328.07,3853.22,141.7


### Entrega 2 - Rankings Comerciais (Performance de Produtos)
**Objetivo:** Identificar os extremos do catálogo de produtos através da função `display()`.
* **Top 5 Mais Vendidos:** Produtos com maior saída em volume absoluto.
* **Top 5 Menos Vendidos:** Produtos com menor tração comercial no histórico.

In [0]:
from pyspark.sql import functions as F

# Preparação da base de dados para o ranking
df_ranking_base = spark.table("silver.fat_itens_pedidos").join(spark.table("silver.dim_produtos"), "id_produto", "inner")

# Agrupamento por produto para contabilizar vendas
df_produtos_agregados = df_ranking_base.groupBy(
    "id_produto", 
    "nome_produto", 
    "categoria_produto"
).agg(
    F.count("id_item").alias("quantidade_vendida")
)

# Ranking 1: Top 5 Produtos mais Vendidos 
# Ordenação decrescente pela quantidade vendida 
df_top_5_mais = df_produtos_agregados.orderBy(F.col("quantidade_vendida").desc()).limit(5)

print("Exibindo: Top 5 Produtos MAIS Vendidos")
display(df_top_5_mais[["nome_produto", "categoria_produto", "quantidade_vendida"]])


# Ranking 2: Top 5 Produtos menos Vendidos
# Ordenação crescente pela quantidade vendida
df_top_5_menos = df_produtos_agregados.orderBy(F.col("quantidade_vendida").asc()).limit(5)

print("Exibindo: Top 5 Produtos MENOS Vendidos")
display(df_top_5_menos[["nome_produto", "categoria_produto", "quantidade_vendida"]])

Exibindo: Top 5 Produtos MAIS Vendidos


nome_produto,categoria_produto,quantidade_vendida
Estante de Livros Luxo,moveis_decoracao,527
Cobertor Cinza,cama_mesa_banho,488
Cortador de Grama Branco,ferramentas_jardim,484
Kit de Ferramentas Ultra,ferramentas_jardim,392
Kit de Ferramentas Master,ferramentas_jardim,388


Exibindo: Top 5 Produtos MENOS Vendidos


nome_produto,categoria_produto,quantidade_vendida
Cortador de Grama,ferramentas_jardim,1
Secador de Cabelo Verde,beleza_saude,1
Cadeira de Escritório Básico,moveis_decoracao,1
Cadeira para Auto Max,bebes,1
Item Básico Luxo,fashion_bolsas_e_acessorios,1


## 2º Projeto: Satisfação de Clientes e Qualidade de Parceiros
**Descrição:** Este projeto visa mensurar a experiência do consumidor final e o desempenho dos parceiros de venda.

**Objetivo:** Fornecer clareza sobre quais categorias, produtos e vendedores estão gerando as melhores e as piores experiências de compra.

### Entrega 1 - Tabela Principal (`gold.fat_avaliacoes_clientes`)
**Regras de Negócio e Granularidade:**
* **Agrupamento:** Realizado por Categoria do Produto, Nome do Vendedor e Estado do Vendedor.
* **Cálculo de Notas:** 
    * **Positivas:** Notas maiores ou iguais a 4.
    * **Negativas:** Notas menores ou iguais a 2.
* **KPI de Satisfação:** Percentual calculado pela razão entre avaliações positivas e o total.

In [0]:
# Carregamento das tabelas Silver necessárias
df_avaliacoes = spark.table("silver.fat_avaliacoes_pedidos")
df_itens = spark.table("silver.fat_itens_pedidos")
df_produtos = spark.table("silver.dim_produtos")
df_vendedores = spark.table("silver.dim_vendedores")

In [0]:
from pyspark.sql import functions as F

# Join para consolidar a visão de Avaliação + Produto + Vendedor
df_consolidado_aval = df_avaliacoes.join(df_itens, "id_pedido", "inner") \
    .join(df_produtos, "id_produto", "inner") \
    .join(df_vendedores, "id_vendedor", "inner")

# Agrupamento e Cálculo de Métricas
df_gold_avaliacoes = df_consolidado_aval.groupBy(
    "categoria_produto",
    "nome_vendedor",
    F.col("estado").alias("estado_vendedor")
).agg(
    F.count("id_avaliacao").alias("total_avaliacoes"),
    F.avg("nota_avaliacao").alias("avaliacao_media"),
    F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
    F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
)

# Cálculo do Percentual de Satisfação e Arredondamentos
df_gold_avaliacoes = df_gold_avaliacoes.withColumn(
    "percentual_satisfacao",
    (F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100
).select(
    "categoria_produto",
    "nome_vendedor",
    "estado_vendedor",
    "total_avaliacoes",
    F.round("avaliacao_media", 2).alias("avaliacao_media"),
    "total_avaliacoes_positivas",
    "total_avaliacoes_negativas",
    F.round("percentual_satisfacao", 2).alias("percentual_satisfacao")
)

# Escrita da tabela na camada Gold
df_gold_avaliacoes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fat_avaliacoes_clientes")

display(df_gold_avaliacoes.limit(5))

categoria_produto,nome_vendedor,estado_vendedor,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
esporte_lazer,Emanuelly Mendonça,SP,393,4.14,311,52,79.13
ferramentas_jardim,Maya Cirino,SP,7,4.43,7,0,100.0
cool_stuff,Milena Moreira,RS,124,4.56,114,5,91.94
pet_shop,Bárbara Pacheco,RJ,97,4.2,77,18,79.38
brinquedos,Isabela Mendes,RJ,355,4.23,291,42,81.97


### Entrega 2 - Rankings de Qualidade (Extremos de Performance)
**Objetivo:** Identificar os produtos e vendedores com melhor e pior avaliação média no marketplace.
* **Critério de Ordenação Rigoroso:** Aplicação de ordenação composta pela Nota Média e, em caso de empate, pelo Volume de Avaliações de forma decrescente para garantir relevância estatística.
* **Resultados:** Exibição do Produto e Vendedor nos topos e bases do ranking.

In [0]:
from pyspark.sql import functions as F

# Preparação da base agregada por produto
df_ranking_produto = spark.table("silver.fat_avaliacoes_pedidos") \
    .join(spark.table("silver.fat_itens_pedidos"), "id_pedido") \
    .join(spark.table("silver.dim_produtos"), "id_produto") \
    .groupBy("id_produto", "nome_produto") \
    .agg(
        F.avg("nota_avaliacao").alias("nota_media"),
        F.count("id_avaliacao").alias("volume_avaliacoes")
    )

# Preparação da base agregada por vendedor
df_ranking_vendedor = spark.table("silver.fat_avaliacoes_pedidos") \
    .join(spark.table("silver.fat_itens_pedidos"), "id_pedido") \
    .join(spark.table("silver.dim_vendedores"), "id_vendedor") \
    .groupBy("id_vendedor", "nome_vendedor") \
    .agg(
        F.avg("nota_avaliacao").alias("nota_media"),
        F.count("id_avaliacao").alias("volume_avaliacoes")
    )

# Produto mais bem avaliado
print("1. O Produto MAIS bem avaliado:")
display(df_ranking_produto.orderBy(F.col("nota_media").desc(), F.col("volume_avaliacoes").desc()).limit(1))

# Produto menos bem avaliado 
print("2. O Produto MENOS bem avaliado:")
display(df_ranking_produto.orderBy(F.col("nota_media").asc(), F.col("volume_avaliacoes").desc()).limit(1))

# Vendedor mais bem avaliado
print("3. O Vendedor MAIS bem avaliado:")
display(df_ranking_vendedor.orderBy(F.col("nota_media").desc(), F.col("volume_avaliacoes").desc()).limit(1))

# Vendedor menos bem avaliado
print("4. O Vendedor MENOS bem avaliado:")
display(df_ranking_vendedor.orderBy(F.col("nota_media").asc(), F.col("volume_avaliacoes").desc()).limit(1))

1. O Produto MAIS bem avaliado:


id_produto,nome_produto,nota_media,volume_avaliacoes
37eb69aca8718e843d897aa7b82f462d,Kit de Ferramentas Dourado,5.0,15


2. O Produto MENOS bem avaliado:


id_produto,nome_produto,nota_media,volume_avaliacoes
0e1fa2aadc04afbf8fb30200aeba06a2,Conjunto de Panelas,1.0,10


3. O Vendedor MAIS bem avaliado:


id_vendedor,nome_vendedor,nota_media,volume_avaliacoes
48efc9d94a9834137efd9ea76b065a38,Luiz Otávio Abreu,5.0,32


4. O Vendedor MENOS bem avaliado:


id_vendedor,nome_vendedor,nota_media,volume_avaliacoes
8d92f3ea807b89465643c219455e7369,Sra. Fernanda Santos,1.0,8
